In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import BatchNormalization

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import seaborn as sns

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
dataset_path = "/content/drive/MyDrive/deepfake_dataset/subset_dataset"

train_dir = dataset_path + "/train"
val_dir = dataset_path + "/validation"
test_dir = dataset_path + "/test"

In [4]:
IMG_SIZE = (300,300)
BATCH_SIZE = 32
EPOCHS = 10
LR = 0.0003

In [5]:
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [6]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 6000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.


In [7]:
base_model = EfficientNetB3(
    weights='imagenet',
    include_top=False,
    input_shape=(300,300,3)
)

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [8]:
for layer in base_model.layers:
    layer.trainable = False


In [9]:
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

from tensorflow.keras.layers import BatchNormalization

x = base_model.output

x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)

output = Dense(1, activation='sigmoid')(x)

In [10]:
from tensorflow.keras.models import Model

model = Model(inputs=base_model.input, outputs=output)

In [11]:
optimizer = Adam(learning_rate=LR)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 300,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 300, 300,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 300, 300,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 300, 300,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 301, 301,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 150, 150,  │      1,080 │ stem_conv_pad[0]… │
│                     │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 150, 150,  │        160 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 150, 150,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 150, 150,  │        360 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 150, 150,  │        160 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 150, 150,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 40)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 40)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 10)  │        410 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 40)  │        440 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 150, 150,  │          0 │ block1a_activati… │
│ (Multiply)          │ 40)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 150, 150,  │        960 │ block1a_se_excit

 Total params: 11,199,664 (42.72 MB)

 Trainable params: 413,057 (1.58 MB)

 Non-trainable params: 10,786,607 (41.15 MB)

In [12]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 2006s 10s/step - accuracy: 0.5735 - loss: 0.8418 - val_accuracy: 0.7440 - val_loss: 0.5850
Epoch 2/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 178s 946ms/step - accuracy: 0.6873 - loss: 0.6374 - val_accuracy: 0.7460 - val_loss: 0.5627
Epoch 3/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 180s 955ms/step - accuracy: 0.7087 - loss: 0.6077 - val_accuracy: 0.7585 - val_loss: 0.5516
Epoch 4/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 180s 955ms/step - accuracy: 0.7219 - loss: 0.5790 - val_accuracy: 0.7525 - val_loss: 0.5567
Epoch 5/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 179s 953ms/step - accuracy: 0.7412 - loss: 0.5626 - val_accuracy: 0.7735 - val_loss: 0.5419


In [13]:
for layer in base_model.layers[-200:]:
    layer.trainable = True

In [14]:
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [15]:
history_finetune = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15
)

Epoch 1/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 336s 1s/step - accuracy: 0.6644 - loss: 0.6224 - val_accuracy: 0.7580 - val_loss: 0.4950
Epoch 2/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 188s 999ms/step - accuracy: 0.7660 - loss: 0.4835 - val_accuracy: 0.8110 - val_loss: 0.4199
Epoch 3/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 190s 1s/step - accuracy: 0.8373 - loss: 0.3739 - val_accuracy: 0.8285 - val_loss: 0.3878
Epoch 4/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 192s 1s/step - accuracy: 0.8624 - loss: 0.3122 - val_accuracy: 0.8430 - val_loss: 0.3613
Epoch 5/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 191s 1s/step - accuracy: 0.8970 - loss: 0.2510 - val_accuracy: 0.8560 - val_loss: 0.3359
Epoch 6/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 189s 1s/step - accuracy: 0.9043 - loss: 0.2361 - val_accuracy: 0.8700 - val_loss: 0.3071
Epoch 7/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 188s 999ms/step - accuracy: 0.9194 - loss: 0.2085 - val_accuracy: 0.8685 - val_loss: 0.3150
Epoch 8/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 187s 994ms/step - accuracy: 0.9332 - loss: 0.1732 -

In [16]:
print(train_data.class_indices)

{'fake': 0, 'real': 1}


In [17]:
test_loss, test_accuracy = model.evaluate(test_data)

print("Test Accuracy:", test_accuracy)

63/63 ━━━━━━━━━━━━━━━━━━━━ 706s 11s/step - accuracy: 0.8582 - loss: 0.3126
Test Accuracy: 0.8450000286102295


In [ ]:
predictions = model.predict(test_data)

y_pred = (predictions > 0.5).astype(int)

y_true = test_data.classes

62/63 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step

In [ ]:
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred)

recall = recall_score(y_true, y_pred)

f1 = f1_score(y_true, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)